# World Maps — generators as simulators: do imagined outcomes predict what people predict? (Colab)

Physion (Bear et al. 2021): 1,144 test clips with per-stimulus human predictions (66–96 participants each) and ground
truth. People saw the first **1.5 s** and answered "will the red object touch the yellow area?". Here a video generator
**continues the 1.5 s context** and a validated red–yellow contact detector reads the imagined outcome. The question is
not only accuracy: do the generator's imagined outcomes agree with people on *which* stimuli are hard (partial
correlation after the label is removed), more than frozen-encoder readouts and a kinematic heuristic do?

| Stage | What | Time (A100) |
|---|---|---|
| Data + detector | 270 MB from S3, labels and human data from GitHub; detector validated on the real outcomes (CPU) | ≈ 8 min |
| Encoders | V-JEPA 2 L/H, VideoMAE-L, DINOv2-L, SigLIP 2 on the 1.5 s context (1,144 clips) | ≈ 15 min |
| VLM | Qwen3-VL P(yes) | ≈ 20 min |
| Generation | LTX-Video continuations for a stratified subset (default 40 per scenario = 320 clips, 1 sample) | ≈ 2–3 h |
| Analysis | humans, detector gate, readouts, generator, paired comparisons, figures | ≈ 5 min |

The last cell downloads `world_maps_physion_results.zip`. Nothing is pushed to GitHub.


In [ ]:
#@title 1. Configuration
MODELS = ["vjepaL", "vjepaH", "vmaeL", "dinoL", "siglip2"]  #@param {type:"raw"}
RUN_VLM = True              #@param {type:"boolean"}
RUN_GENERATION = True       #@param {type:"boolean"}
GENERATOR = "ltx"           #@param ["ltx", "cosmos"]
PER_SCENARIO = 40           #@param {type:"integer"}   # stimuli per scenario (balanced on the label); 8 scenarios
K_SAMPLES = 1               #@param {type:"integer"}
GEN_SIZE = 512              #@param {type:"integer"}   # multiple of 32
N_FRAMES = 97               #@param {type:"integer"}   # 8k+1 for LTX
SMOKE_TEST_FIRST = True     #@param {type:"boolean"}
MOUNT_DRIVE = False         #@param {type:"boolean"}

import os, torch
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch to a GPU runtime)'}  {gpu_gb:.0f} GB")
VLM_ID = "Qwen/Qwen3-VL-8B-Instruct" if gpu_gb >= 20 else "Qwen/Qwen3-VL-4B-Instruct"
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok: os.environ["HF_TOKEN"] = tok; print("HF_TOKEN loaded from Colab secrets")
except Exception:
    pass


In [ ]:
#@title 2. Code and dependencies (about 2 min)
%cd /content
import os
if not os.path.exists("world-maps"):
    !git clone -q https://github.com/physicalreasoning/world-maps.git
%cd /content/world-maps
!git pull -q; git rev-parse --short HEAD
!pip install -q "transformers>=5.0" accelerate "diffusers>=0.33" imageio imageio-ffmpeg matplotlib pillow opencv-python-headless scikit-learn scipy pandas safetensors
import torch, transformers, diffusers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| diffusers", diffusers.__version__)
os.environ["PYTHONPATH"] = "/content/world-maps/src"
os.makedirs("results/physion", exist_ok=True)


In [ ]:
%%writefile /content/world-maps/src/world_maps/physion.py
"""Physion (Bear et al. 2021): object-contact prediction with per-stimulus human predictions.

Assets (public, fetched at run time):
  PhysionTest-Core zip (270 MB, S3): per scenario ``mp4s-redyellow`` (agent red, patient yellow; 512 px,
  30 fps, 5 s or longer; the outcome is shown), ``mp4s`` (random colours, shown to people), ``maps``
  (first-frame masks).
  Repository cogtoolslab/physics-benchmarking-neurips2021: ground-truth labels and the per-stimulus human
  accuracy and participant counts (1,144 of the 1,200 test stimuli).

Participants saw the first 1.5 s (45 frames) and answered "will the red object touch the yellow area?".
Here ``context`` = the first 45 frames (what people saw); ``outcome`` = the rest (ground truth for the
contact detector). Human P(yes) = accuracy when the label is True, 1 - accuracy when it is False.
"""

from __future__ import annotations

import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from .paths import DATA

PHYSION = DATA / "physion"
CORE_URL = "https://physics-benchmarking-neurips2021-dataset.s3.amazonaws.com/Physion.zip"
REPO_URL = "https://github.com/cogtoolslab/physics-benchmarking-neurips2021"
SCENARIOS = {
    "dominoes": "Dominoes",
    "support": "Support",
    "collide": "Collide",
    "contain": "Contain",
    "drop": "Drop",
    "roll": "Roll",
    "link": "Link",
    "drape": "Drape",
}
HUMAN_FILES = {
    "dominoes": "dominoes_pilot-production_1",
    "support": "towers_pilot-production_2",
    "collide": "collision_pilot-production_2",
    "contain": "containment_pilot-production_2",
    "drop": "drop_pilot-production_2",
    "roll": "rollingsliding_pilot-production_2",
    "link": "linking_pilot-production_2",
    "drape": "clothiness_pilot-production_2",
}
FPS = 30
CONTEXT_FRAMES = 45  # 1.5 s shown to participants


def fetch_assets() -> None:
    PHYSION.mkdir(parents=True, exist_ok=True)
    if not (PHYSION / "Physion").exists():
        z = PHYSION / "Physion.zip"
        if not z.exists():
            subprocess.run(["curl", "-L", "--retry", "3", "-o", str(z), CORE_URL], check=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(PHYSION, members=[m for m in zf.namelist() if not m.startswith("__MACOSX")])
    if not (PHYSION / "repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(PHYSION / "repo")], check=True)


@dataclass
class Stim:
    stim_id: str
    scenario: str
    label: bool
    human_acc: float | None
    n_human: int | None

    @property
    def p_yes_human(self) -> float | None:
        if self.human_acc is None:
            return None
        return self.human_acc if self.label else 1.0 - self.human_acc

    @property
    def video(self) -> Path:
        head, idx = self.stim_id.rsplit("_", 1)
        return (
            PHYSION
            / "Physion"
            / SCENARIOS[self.scenario]
            / "mp4s-redyellow"
            / f"{head}-redyellow_{idx}_img.mp4"
        )

    @property
    def map_png(self) -> Path:
        return PHYSION / "Physion" / SCENARIOS[self.scenario] / "maps" / f"{self.stim_id}_map.png"


def stimuli(with_human_only: bool = False) -> list[Stim]:
    repo = PHYSION / "repo"
    labels = pd.read_csv(repo / "stimuli" / "stimulus_names_and_labels.csv")
    human = {}
    for fname in HUMAN_FILES.values():
        f = repo / "results" / "csv" / "humans" / f"human_accuracy-{fname}_testing.csv"
        if f.exists():
            for _, r in pd.read_csv(f).iterrows():
                human[r.stim_ID] = (float(r.correct), int(r.c))
    out = []
    for _, r in labels.iterrows():
        h = human.get(r.stim_ID)
        s = Stim(
            stim_id=r.stim_ID,
            scenario=r.scenario,
            label=bool(r.label) if not isinstance(r.label, str) else r.label == "True",
            human_acc=h[0] if h else None,
            n_human=h[1] if h else None,
        )
        if with_human_only and h is None:
            continue
        out.append(s)
    return out


def stim_table(stims: list[Stim]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "stim_id": s.stim_id,
                "scenario": s.scenario,
                "label": s.label,
                "human_acc": s.human_acc,
                "n_human": s.n_human,
                "p_yes_human": s.p_yes_human,
            }
            for s in stims
        ]
    )


def read_frames(path: Path, start: int = 0, stop: int | None = None, size: int | None = None) -> np.ndarray:
    """RGB uint8 frames [start, stop) of a video, optionally resized to size x size."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    i = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if i >= start and (stop is None or i < stop):
            fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            if size and fr.shape[0] != size:
                fr = cv2.resize(fr, (size, size), interpolation=cv2.INTER_AREA)
            frames.append(fr)
        i += 1
        if stop is not None and i >= stop:
            break
    cap.release()
    return np.stack(frames) if frames else np.zeros((0, size or 512, size or 512, 3), np.uint8)


def sample_indices(n: int, k: int) -> np.ndarray:
    return np.linspace(0, n - 1, k).round().astype(int)


# ----------------------------------------------------------------------------- contact detector


class ColourModel:
    """Per-stimulus HSV model of the red agent and yellow patient.

    Calibrated on the first frame with the released segmentation map.
    """

    def __init__(self, frame0_rgb: np.ndarray, map_rgb: np.ndarray | None = None):
        hsv = cv2.cvtColor(frame0_rgb, cv2.COLOR_RGB2HSV).astype(int)
        if map_rgb is not None and map_rgb.shape[:2] != frame0_rgb.shape[:2]:
            map_rgb = cv2.resize(map_rgb, frame0_rgb.shape[1::-1], interpolation=cv2.INTER_NEAREST)
        if map_rgb is not None:
            red = (map_rgb[..., 0] > 200) & (map_rgb[..., 1] < 60) & (map_rgb[..., 2] < 60)
            yel = (map_rgb[..., 0] > 200) & (map_rgb[..., 1] > 200) & (map_rgb[..., 2] < 80)
        else:
            red = yel = np.zeros(frame0_rgb.shape[:2], bool)
        self.red = self._stats(hsv, red, default=(177, 230, 100))
        self.yellow = self._stats(hsv, yel, default=(26, 110, 210))
        self.map_yellow = yel if yel.sum() > 20 else None

    @staticmethod
    def _stats(hsv, mask, default):
        if mask.sum() < 20:
            return {
                "h": default[0],
                "s": default[1],
                "v": default[2],
                "s_lo": max(15, default[1] - 80),
                "v_lo": max(30, default[2] - 90),
            }
        h, s_, v = hsv[mask][:, 0], hsv[mask][:, 1], hsv[mask][:, 2]
        hm = int(np.median(h)) if default[0] < 90 else int(np.median(np.where(h < 90, h + 180, h))) % 180
        return {
            "h": hm,
            "s": int(np.median(s_)),
            "v": int(np.median(v)),
            "s_lo": int(max(15, np.percentile(s_, 3) - 15)),
            "v_lo": int(max(25, np.percentile(v, 3) - 25)),
        }

    def masks(self, frame_rgb: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV).astype(int)
        h, s_, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]

        def near(h0, tol):
            d = np.abs(h - h0)
            return np.minimum(d, 180 - d) <= tol

        red = near(self.red["h"], 8) & (s_ >= max(self.red["s_lo"], 90)) & (v >= self.red["v_lo"])
        yellow = (
            near(self.yellow["h"], 5)
            & (s_ >= self.yellow["s_lo"])
            & (v >= self.yellow["v_lo"])
            & (np.abs(v - self.yellow["v"]) <= 90)
        )
        k = np.ones((3, 3), np.uint8)
        red = cv2.morphologyEx(red.astype(np.uint8), cv2.MORPH_OPEN, k) > 0
        yellow = cv2.morphologyEx(yellow.astype(np.uint8), cv2.MORPH_OPEN, k) > 0
        return red, yellow


def frame_measures(
    frame_rgb: np.ndarray, cm: ColourModel, dilate_px: int = 2, zone: np.ndarray | None = None
) -> dict:
    """Red-yellow geometry in one frame.

    Minimum distance (px at this resolution) and the fraction of the red object inside the dilated yellow
    region. ``zone`` (the yellow region seen during the context) is used in addition to the live yellow
    mask, because the red object hides the part of the zone it lands on.
    """
    red, yellow = cm.masks(frame_rgb)
    out = {
        "red_px": int(red.sum()),
        "yellow_px": int(yellow.sum()),
        "dist": float("nan"),
        "overlap": float("nan"),
    }
    ref = yellow if zone is None else (yellow | zone)
    if red.sum() < 10 or ref.sum() < 10:
        return out
    yd = cv2.dilate(ref.astype(np.uint8), np.ones((2 * dilate_px + 1,) * 2, np.uint8)) > 0
    inter = (red & yd).sum()
    # fraction of the red object inside the zone, or of the zone covered by red (a cloth covers a small zone)
    out["overlap"] = float(max(inter / red.sum(), inter / max(ref.sum(), 1)))
    if (red & ref).any():
        out["dist"] = 0.0
    else:
        dist = cv2.distanceTransform((~ref).astype(np.uint8), cv2.DIST_L2, 3)
        out["dist"] = float(dist[red].min())
    return out


def context_zone(context: np.ndarray, cm: ColourModel) -> np.ndarray:
    """Union of the yellow masks over the context frames (the patient region before anything happens)."""
    z = np.zeros(context.shape[1:3], bool)
    for f in context[:: max(1, len(context) // 10)]:
        z |= cm.masks(f)[1]
    return z


def contact_score(
    frames: np.ndarray,
    cm: ColourModel,
    context: np.ndarray | None = None,
    touch_px: float = 2.0,
    overlap_gain: float = 0.10,
) -> dict:
    """Contact decision for an outcome sequence, relative to the end of the context (if given).

    contact = the red object comes within ``touch_px`` of the yellow region while it was farther than that
    at the end of the context, or its overlap with the yellow region grows by more than ``overlap_gain`` of
    its area. Distances are expressed at 512 px. Returns the decision, a soft score (overlap gain and
    distance drop), and coverage diagnostics.
    """
    scale = frames.shape[1] / 512.0
    zone = context_zone(context, cm) if context is not None and len(context) else None
    if zone is not None and zone.shape != frames.shape[1:3]:
        zone = cv2.resize(zone.astype(np.uint8), frames.shape[1:3][::-1], interpolation=cv2.INTER_NEAREST) > 0
    m = [frame_measures(f, cm, zone=zone) for f in frames]
    d = np.array([x["dist"] for x in m]) / scale
    ov = np.array([x["overlap"] for x in m])
    valid = ~np.isnan(d)
    base_d, base_ov = float("inf"), 0.0
    if context is not None and len(context):
        mc = [frame_measures(f, cm, zone=zone) for f in context[-10:]]
        dc = np.array([x["dist"] for x in mc]) / scale
        oc = np.array([x["overlap"] for x in mc])
        base_d = float(np.nanmin(dc)) if np.isfinite(dc).any() else float("inf")
        base_ov = float(np.nanmax(oc)) if np.isfinite(oc).any() else 0.0
    if not valid.any():
        return {
            "contact": None,
            "min_dist": float("nan"),
            "max_overlap": float("nan"),
            "frac_valid": 0.0,
            "score": float("nan"),
            "base_dist": base_d,
            "base_overlap": base_ov,
            "vanish": False,
            "first_contact_frame": None,
        }
    dm, om = float(np.nanmin(d)), float(np.nanmax(ov))
    touch = dm <= touch_px and (not np.isfinite(base_d) or base_d > touch_px)
    gain = om - base_ov > overlap_gain
    # the red object vanishes inside the zone (a container hides it): its area drops while it sits on the zone
    red_px = np.array([x["red_px"] for x in m], float)
    base_red = (
        float(np.mean([x["red_px"] for x in mc])) if context is not None and len(context) else float("nan")
    )
    vanish = bool(
        np.isfinite(base_red)
        and base_red > 50
        and red_px.min() < 0.4 * base_red
        and np.nanmax(ov[: int(np.argmin(red_px)) + 1]) > 0.5
    )
    contact = bool(touch or gain or vanish)
    hit = np.where(valid & ((d <= touch_px) | (ov - base_ov > overlap_gain)))[0]
    score = float(
        max(om - base_ov, 0.0) + max((min(base_d, 60.0) if np.isfinite(base_d) else 60.0) - dm, 0.0) / 60.0
    )
    return {
        "contact": contact,
        "min_dist": dm,
        "max_overlap": om,
        "frac_valid": float(valid.mean()),
        "score": score + (0.5 if vanish else 0.0),
        "base_dist": base_d,
        "base_overlap": base_ov,
        "vanish": vanish,
        "first_contact_frame": int(hit[0]) if len(hit) else None,
    }


def kinematic_features(frames: np.ndarray, cm: ColourModel) -> np.ndarray:
    """Heuristic features from the context frames.

    Last distance, distance change, approach speed toward the yellow region, red speed, mask areas and the
    relative offset. Used by the kinematic baseline readout.
    """
    ds, cr, cy = [], [], []
    for fr in frames:
        r, y = cm.masks(fr)
        ds.append(frame_measures(fr, cm)["dist"])
        cr.append(np.array(np.where(r)).mean(1)[::-1] if r.any() else np.array([np.nan, np.nan]))
        cy.append(np.array(np.where(y)).mean(1)[::-1] if y.any() else np.array([np.nan, np.nan]))
    ds, cr, cy = np.array(ds), np.array(cr), np.array(cy)
    last = np.nanmean(ds[-5:]) if np.isfinite(ds[-5:]).any() else 512.0
    first = np.nanmean(ds[:5]) if np.isfinite(ds[:5]).any() else 512.0
    v = np.nanmean(np.diff(cr, axis=0)[-10:], axis=0) if len(cr) > 10 else np.zeros(2)
    to_y = np.nan_to_num(cy[-1] - cr[-1])
    approach = float(np.dot(np.nan_to_num(v), to_y / (np.linalg.norm(to_y) + 1e-6)))
    r_last, y_last = cm.masks(frames[-1])
    return np.nan_to_num(
        np.array(
            [
                last,
                first - last,
                approach,
                float(np.linalg.norm(np.nan_to_num(v))),
                r_last.mean(),
                y_last.mean(),
                to_y[0],
                to_y[1],
            ],
            float,
        ),
        nan=0.0,
    )


In [ ]:
%%writefile /content/world-maps/scripts/physion_extract.py
"""Physion: measurement baselines and model readouts on the 1.5 s context that people saw.

Stages (``--stages``):
  detector   run the red-yellow contact detector on the REAL outcome frames of every stimulus (validation of the
             measurement against the ground-truth label) and the kinematic heuristic features of the context
             -> results/physion/detector_real.csv, results/physion/kinematics.csv            (CPU, about 5 min)
  encoders   frozen video/image encoders on the 1.5 s context (16 frames) -> results/physion/features/{model}.npz
  vlm        Qwen3-VL P(yes) to "will the red object touch the yellow area?" -> results/physion/vlm.csv

Models: vjepaL, vjepaH, vmaeL, dinoL, siglip2 (mean over 8 frames + last frame).
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from world_maps import physion as P  # noqa: E402
from world_maps.paths import RESULTS  # noqa: E402

OUT = RESULTS / "physion"
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
MODELS = {
    "vjepaL": ("vjepa2", "facebook/vjepa2-vitl-fpc64-256", 256),
    "vjepaH": ("vjepa2", "facebook/vjepa2-vith-fpc64-256", 256),
    "vmaeL": ("videomae", "MCG-NJU/videomae-large", 224),
    "vmaeB": ("videomae", "MCG-NJU/videomae-base", 224),
    "dinoL": ("image", "facebook/dinov2-large", 224),
    "dinoB": ("image", "facebook/dinov2-base", 224),
    "siglip2": ("image", "google/siglip2-so400m-patch16-256", 256),
}
VLM_ID = "Qwen/Qwen3-VL-8B-Instruct"
PROMPT = ("This short clip shows a physical scene. One object is red and there is a yellow area or object. The clip stops before "
          "the outcome. Based on how things are moving, will the red object touch the yellow area or object after the clip ends? "
          "Answer Yes or No.")


def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))


def context_frames(stim: P.Stim, size: int, k: int = 16) -> np.ndarray:
    fr = P.read_frames(stim.video, 0, P.CONTEXT_FRAMES, size=size)
    return fr[P.sample_indices(len(fr), k)]


def pool_grid(tok, side, grid=4):
    B, N, D = tok.shape
    x = tok.view(B, side, side, D).permute(0, 3, 1, 2)
    if side % grid and x.device.type == "mps":
        x = x.cpu()
    return F.adaptive_avg_pool2d(x, grid).permute(0, 2, 3, 1).reshape(B, -1)


# ----------------------------------------------------------------------------- detector + kinematics (CPU)

def stage_detector(stims, size=256, log=print):
    rows, kin = [], []
    t0 = time.time()
    for i, s in enumerate(stims):
        fr = P.read_frames(s.video, size=size)
        if len(fr) <= P.CONTEXT_FRAMES + 5:
            continue
        m = cv2.imread(str(s.map_png))
        cm = P.ColourModel(fr[0], cv2.cvtColor(m, cv2.COLOR_BGR2RGB) if m is not None else None)
        ctx, out = fr[: P.CONTEXT_FRAMES], fr[P.CONTEXT_FRAMES:]
        r = P.contact_score(out, cm, context=ctx)
        r_ctx = P.contact_score(ctx[-15:], cm, context=ctx[:30])
        rows.append({"stim_id": s.stim_id, "scenario": s.scenario, "label": s.label, "p_yes_human": s.p_yes_human, "n_human": s.n_human,
                     "n_frames": len(fr), **{f"real_{k}": v for k, v in r.items()}, "ctx_contact": r_ctx["contact"], "ctx_score": r_ctx["score"]})
        kin.append({"stim_id": s.stim_id, **{f"k{j}": v for j, v in enumerate(P.kinematic_features(ctx[::3], cm))}})
        if i % 100 == 0:
            log(f"  detector {i}/{len(stims)} ({time.time() - t0:.0f}s)")
    pd.DataFrame(rows).to_csv(OUT / "detector_real.csv", index=False)
    pd.DataFrame(kin).to_csv(OUT / "kinematics.csv", index=False)
    df = pd.DataFrame(rows)
    acc = df.groupby("scenario").apply(lambda g: float((g.real_contact == g.label).mean()))
    log("detector accuracy on real outcomes by scenario:\n" + acc.round(2).to_string())


# ----------------------------------------------------------------------------- encoders

def stage_encoders(stims, names, log=print):
    (OUT / "features").mkdir(parents=True, exist_ok=True)
    meta_p = OUT / "features" / "models.json"
    meta = json.load(open(meta_p)) if meta_p.exists() else {}
    for name in names:
        family, hub, size = MODELS[name]
        f = OUT / "features" / f"{name}.npz"
        if f.exists():
            log(f"{name}: done, skipping")
            continue
        t0 = time.time()
        try:
            feats = {}
            if family == "vjepa2":
                from transformers import VJEPA2Model

                m = VJEPA2Model.from_pretrained(hub).to(DEV).eval()
                dtype = torch.bfloat16 if DEV == "cuda" else torch.float32
                m = m.to(dtype)
                side = size // 16
                per = side * side
                rows_mean, rows_last, rows_grid = [], [], []
                for i, s in enumerate(stims):
                    x = torch.from_numpy(context_frames(s, size)).permute(0, 3, 1, 2).float() / 255.0
                    x = ((x - MEAN) / STD).unsqueeze(0).to(DEV, dtype)
                    with torch.no_grad():
                        h = m(x, skip_predictor=True).last_hidden_state.float()
                    rows_mean.append(h.mean(1).cpu().half().numpy()[0])
                    rows_last.append(h[:, -per:].mean(1).cpu().half().numpy()[0])
                    rows_grid.append(pool_grid(h[:, -per:], side).cpu().half().numpy()[0])
                    if i % 200 == 0:
                        log(f"  {name} {i}/{len(stims)}")
                feats = {"mean": np.stack(rows_mean), "last_slot": np.stack(rows_last), "last_grid": np.stack(rows_grid)}
            elif family == "videomae":
                from transformers import VideoMAEModel

                try:  # the fused attention kernel has raised illegal-memory-access errors for this model on some CUDA stacks
                    m = VideoMAEModel.from_pretrained(hub, attn_implementation="eager").to(DEV).eval()
                except TypeError:
                    m = VideoMAEModel.from_pretrained(hub).to(DEV).eval()
                side = size // 16
                per = side * side
                rows_mean, rows_grid = [], []
                for i, s in enumerate(stims):
                    x = torch.from_numpy(context_frames(s, size)).permute(0, 3, 1, 2).float() / 255.0
                    x = ((x - MEAN) / STD).unsqueeze(0).to(DEV)
                    with torch.no_grad():
                        h = m(pixel_values=x).last_hidden_state.float()
                    rows_mean.append(h.mean(1).cpu().half().numpy()[0])
                    rows_grid.append(pool_grid(h[:, -per:], side).cpu().half().numpy()[0])
                    if i % 200 == 0:
                        log(f"  {name} {i}/{len(stims)}")
                feats = {"mean": np.stack(rows_mean), "last_grid": np.stack(rows_grid)}
            else:
                from transformers import AutoModel

                m = AutoModel.from_pretrained(hub).to(DEV).eval()
                vision = m.vision_model if hasattr(m, "vision_model") else m
                mean, std = (torch.tensor([0.5] * 3).view(1, 3, 1, 1),) * 2 if "siglip" in hub else (MEAN, STD)
                rows_mean, rows_last, rows_diff = [], [], []
                for i, s in enumerate(stims):
                    fr = context_frames(s, size, k=8)
                    x = ((torch.from_numpy(fr).permute(0, 3, 1, 2).float() / 255.0 - mean) / std).to(DEV)
                    with torch.no_grad():
                        out = vision(pixel_values=x)
                        e = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]
                        e = e.float()
                    rows_mean.append(e.mean(0).cpu().half().numpy())
                    rows_last.append(e[-1].cpu().half().numpy())
                    rows_diff.append((e[-1] - e[0]).cpu().half().numpy())
                    if i % 200 == 0:
                        log(f"  {name} {i}/{len(stims)}")
                feats = {"mean": np.stack(rows_mean), "last": np.stack(rows_last), "last_minus_first": np.stack(rows_diff)}
            np.savez(f, stim_id=np.array([s.stim_id for s in stims]), **{f"X_{k}": v for k, v in feats.items()})
            meta[name] = {"status": "ok", "hub": hub, "params": n_params(m), "seconds": round(time.time() - t0, 1), "n": len(stims),
                          "dims": {k: int(v.shape[1]) for k, v in feats.items()}}
            del m
            if DEV == "cuda":
                torch.cuda.empty_cache()
        except Exception as e:
            import traceback
            traceback.print_exc()
            meta[name] = {"status": f"failed: {type(e).__name__}: {str(e)[:200]}", "hub": hub}
            json.dump(meta, open(meta_p, "w"), indent=1)
            if "CUDA" in str(e) or "cuda" in type(e).__name__.lower():
                log(f"{name}: CUDA error; the CUDA context is unusable in this process. Rerun the stage: finished models are skipped.")
                break
        json.dump(meta, open(meta_p, "w"), indent=1)
        log(f"{name}: {meta[name]['status']}")


# ----------------------------------------------------------------------------- VLM

def stage_vlm(stims, hub=VLM_ID, log=print):
    from transformers import AutoModelForImageTextToText, AutoProcessor

    dtype = torch.bfloat16 if DEV != "cpu" else torch.float32
    try:
        m = AutoModelForImageTextToText.from_pretrained(hub, dtype=dtype, device_map={"": DEV}).eval()
    except TypeError:
        m = AutoModelForImageTextToText.from_pretrained(hub, torch_dtype=dtype, device_map={"": DEV}).eval()
    proc = AutoProcessor.from_pretrained(hub)
    tok = proc.tokenizer
    yes_ids = {tok.encode(t, add_special_tokens=False)[0] for t in ("Yes", " Yes", "yes")}
    no_ids = {tok.encode(t, add_special_tokens=False)[0] for t in ("No", " No", "no")}
    fps = 16 / 1.5
    rows = []
    out_f = OUT / "vlm.csv"
    done = set(pd.read_csv(out_f).stim_id) if out_f.exists() else set()
    for i, s in enumerate(stims):
        if s.stim_id in done:
            continue
        frames = context_frames(s, 256)
        messages = [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": PROMPT}]}]
        text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        try:
            inputs = proc(text=[text], videos=[frames], return_tensors="pt",
                          video_metadata=[{"fps": fps, "total_num_frames": len(frames), "duration": len(frames) / fps}])
        except TypeError:
            inputs = proc(text=[text], videos=[frames], return_tensors="pt")
        inputs = {k: (v.to(DEV) if hasattr(v, "to") else v) for k, v in inputs.items()}
        with torch.no_grad():
            logits = m(**inputs).logits[0, -1].float()
        ly, ln = torch.logsumexp(logits[list(yes_ids)], 0), torch.logsumexp(logits[list(no_ids)], 0)
        rows.append({"stim_id": s.stim_id, "p_yes": float(torch.softmax(torch.stack([ly, ln]), 0)[0])})
        if i % 100 == 0:
            log(f"  vlm {i}/{len(stims)}")
            pd.concat([pd.read_csv(out_f) if out_f.exists() else pd.DataFrame(), pd.DataFrame(rows)]).to_csv(out_f, index=False)
            rows = []
    pd.concat([pd.read_csv(out_f) if out_f.exists() else pd.DataFrame(), pd.DataFrame(rows)]).to_csv(out_f, index=False)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--stages", nargs="+", default=["detector", "encoders"])
    ap.add_argument("--models", nargs="+", default=["vjepaL", "vmaeL", "dinoL", "siglip2"])
    ap.add_argument("--vlm-id", default=VLM_ID)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--force", action="store_true", help="recompute the detector stage even if its outputs exist")
    args = ap.parse_args(argv)
    OUT.mkdir(parents=True, exist_ok=True)
    P.fetch_assets()
    stims = P.stimuli(with_human_only=True)
    if args.limit:
        stims = stims[: args.limit]
    P.stim_table(stims).to_csv(OUT / "stimuli.csv", index=False)
    if "detector" in args.stages:
        if (OUT / "detector_real.csv").exists() and (OUT / "kinematics.csv").exists() and not args.force and not args.limit:
            print("detector outputs exist (committed in the repository; same code and data); use --force to recompute")
        else:
            stage_detector(stims)
    if "encoders" in args.stages:
        stage_encoders(stims, args.models)
    if "vlm" in args.stages:
        stage_vlm(stims, args.vlm_id)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/world-maps/scripts/physion_generate.py
"""Physion: generators as simulators. Continue the 1.5 s context with a video generator and read the imagined outcome.

For a stratified subset of stimuli (``--per-scenario`` per scenario, balanced on the label), the generator receives the
context frames people saw and produces the following seconds; the red-yellow contact detector (calibrated on the real
first frame and the real context) is run on the generated frames. Output: results/physion/generated/{generator}/scores.csv
with one row per (stimulus, sample), the detector's decision and soft score, plus a few mp4s and last frames for
inspection.

Generators (diffusers):
  ltx     LTX-Video 2B (Lightricks/LTX-Video-0.9.5 by default; --ltx-id selects another checkpoint, the 0.9.7/0.9.8
          "dev" checkpoints are 13B and need an 80 GB card) with video conditioning (LTXConditionPipeline): the context
          frames are the condition at frame 0; 97 frames at 24 fps are generated (about 2.7 s beyond the context).
          About 15-30 s per clip on an A100 for the 2B model.
  cosmos  nvidia/Cosmos-Predict2-2B-Video2World (video-conditioned world model; experimental in this script).
The prompt describes a camera-fixed physical scene and never states the outcome.
"""

from __future__ import annotations

import argparse
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from world_maps import physion as P  # noqa: E402
from world_maps.paths import RESULTS  # noqa: E402

OUT = RESULTS / "physion" / "generated"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LTX_ID = "Lightricks/LTX-Video-0.9.5"  # 2B; fits a 40 GB card with room to spare
PROMPT = ("A fixed camera films a simple indoor physics scene with a few rigid objects on the floor. The objects keep moving "
          "according to gravity, momentum and collisions, exactly as they were moving. Realistic rendering, no camera motion, "
          "no new objects, no cuts.")
NEGATIVE = "camera motion, zoom, cut, text, watermark, new objects, people, hands, blur, flicker"


def select(stims, per_scenario: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    out = []
    for scen in P.SCENARIOS:
        sub = [s for s in stims if s.scenario == scen]
        for lab in (True, False):
            pool = [s for s in sub if s.label == lab]
            idx = rng.permutation(len(pool))[: per_scenario // 2]
            out += [pool[i] for i in sorted(idx)]
    return out


def load_pipe(kind: str, ltx_id: str = LTX_ID):
    if kind == "ltx":
        from diffusers import LTXConditionPipeline

        pipe = LTXConditionPipeline.from_pretrained(ltx_id, torch_dtype=torch.bfloat16).to(DEV)
        for fn in ("enable_tiling", "enable_slicing"):
            if hasattr(pipe.vae, fn):
                getattr(pipe.vae, fn)()
        n = sum(p.numel() for p in pipe.transformer.parameters()) / 1e9
        print(f"LTX transformer: {n:.1f}B parameters ({ltx_id})", flush=True)
        return pipe
    if kind == "cosmos":
        from diffusers import Cosmos2VideoToWorldPipeline

        return Cosmos2VideoToWorldPipeline.from_pretrained("nvidia/Cosmos-Predict2-2B-Video2World", torch_dtype=torch.bfloat16).to(DEV)
    raise ValueError(kind)


def continue_video(pipe, kind: str, context_rgb: np.ndarray, seed: int, size: int, n_frames: int, steps: int) -> np.ndarray:
    """Return generated frames (T, size, size, 3) uint8 INCLUDING the conditioning part at the start (LTX)."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    frames = [Image.fromarray(f).resize((size, size), Image.BICUBIC) for f in context_rgb]
    if kind == "ltx":
        from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition

        n_cond = 8 * ((len(frames) - 1) // 8) + 1  # LTX wants 8k+1 conditioning frames
        cond = LTXVideoCondition(video=frames[-n_cond:], frame_index=0)
        out = pipe(conditions=[cond], prompt=PROMPT, negative_prompt=NEGATIVE, width=size, height=size, num_frames=n_frames,
                   num_inference_steps=steps, generator=g).frames[0]
    else:
        out = pipe(video=frames[-9:], prompt=PROMPT, negative_prompt=NEGATIVE, height=size, width=size, num_frames=n_frames,
                   num_inference_steps=steps, generator=g).frames[0]
    arr = [np.array(f.convert("RGB")) if isinstance(f, Image.Image) else (f * 255).astype(np.uint8) if f.dtype != np.uint8 else f for f in out]
    return np.stack(arr)


def write_mp4(frames, path, fps=24):
    h, w = frames.shape[1:3]
    vw = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames:
        vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    vw.release()


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--generator", default="ltx", choices=["ltx", "cosmos"])
    ap.add_argument("--per-scenario", type=int, default=40)
    ap.add_argument("--k", type=int, default=1, help="samples per stimulus")
    ap.add_argument("--size", type=int, default=512)
    ap.add_argument("--n-frames", type=int, default=97)
    ap.add_argument("--steps", type=int, default=25)
    ap.add_argument("--ltx-id", default=LTX_ID)
    ap.add_argument("--context-fps", type=float, default=24.0, help="context frames are resampled to the generator's frame rate")
    ap.add_argument("--keep-videos", type=int, default=16)
    ap.add_argument("--limit", type=int, default=None, help="first N selected stimuli (smoke test)")
    ap.add_argument("--out", default=str(OUT))
    args = ap.parse_args(argv)
    P.fetch_assets()
    stims = select(P.stimuli(with_human_only=True), args.per_scenario)
    if args.limit:
        stims = stims[: args.limit]
    out = __import__("pathlib").Path(args.out) / args.generator
    (out / "frames").mkdir(parents=True, exist_ok=True)
    scores_f = out / "scores.csv"
    done = set()
    if scores_f.exists():
        d = pd.read_csv(scores_f)
        done = set(zip(d.stim_id, d["sample"]))
    pipe = load_pipe(args.generator, args.ltx_id)
    t0 = time.time()
    n_kept = 0
    rows = []
    for i, s in enumerate(stims):
        full = P.read_frames(s.video, size=args.size)
        ctx = full[: P.CONTEXT_FRAMES]
        m = cv2.imread(str(s.map_png))
        cm = P.ColourModel(full[0], cv2.cvtColor(m, cv2.COLOR_BGR2RGB) if m is not None else None)
        # resample the 30 fps context to the generator's frame rate
        n_ctx = int(round(P.CONTEXT_FRAMES / P.FPS * args.context_fps))
        ctx_rs = ctx[P.sample_indices(len(ctx), n_ctx)]
        for k in range(args.k):
            if (s.stim_id, k) in done:
                continue
            try:
                gen = continue_video(pipe, args.generator, ctx_rs, seed=10 * i + k, size=args.size, n_frames=args.n_frames, steps=args.steps)
                n_cond = 8 * ((len(ctx_rs) - 1) // 8) + 1 if args.generator == "ltx" else 9
                future = gen[n_cond:] if len(gen) > n_cond + 4 else gen
                r = P.contact_score(future, cm, context=ctx)
                rows.append({"stim_id": s.stim_id, "scenario": s.scenario, "label": s.label, "p_yes_human": s.p_yes_human, "sample": k,
                             "n_generated": int(len(future)), **{f"gen_{kk}": v for kk, v in r.items()}})
                Image.fromarray(future[-1]).resize((256, 256)).save(out / "frames" / f"{s.stim_id}_{k}_last.png")
                if n_kept < args.keep_videos:
                    write_mp4(gen, out / f"{s.stim_id}_{k}.mp4")
                    n_kept += 1
            except Exception as e:
                print(f"  failed {s.stim_id} sample {k}: {type(e).__name__}: {str(e)[:160]}", flush=True)
        if rows and (i % 5 == 0 or i == len(stims) - 1):
            prev = pd.read_csv(scores_f) if scores_f.exists() else pd.DataFrame()
            pd.concat([prev, pd.DataFrame(rows)]).to_csv(scores_f, index=False)
            rows = []
            print(f"{args.generator}: {i + 1}/{len(stims)} stimuli, {time.time() - t0:.0f}s", flush=True)
    if rows:
        prev = pd.read_csv(scores_f) if scores_f.exists() else pd.DataFrame()
        pd.concat([prev, pd.DataFrame(rows)]).to_csv(scores_f, index=False)
    print(f"done in {time.time() - t0:.0f}s -> {scores_f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/world-maps/scripts/physion_analyze.py
"""Physion: do imagined outcomes from video generators predict what people predict?

Readouts, all giving a P(yes) per stimulus for "will the red object touch the yellow area":
  humans       P(yes) from the released per-stimulus accuracy and the ground-truth label (66-96 participants each)
  kinematic    logistic readout on hand-made context features (distance, approach speed, offsets): the appearance /
               extrapolation heuristic
  encoders     logistic readout on frozen video/image features of the context (within-scenario CV and
               leave-one-scenario-out)
  vlm          Qwen3-VL yes/no probability
  generator    the contact detector applied to video continuations (mean over samples), with a per-scenario
               calibration of the detector fitted on REAL outcome frames (measurement, not on human data)

Metrics: accuracy vs ground truth; agreement with humans (Pearson r with human P(yes), Cohen's kappa with the human
majority); error-pattern agreement = partial correlation after regressing the label out of both (does the model get
the same stimuli wrong?); accuracy-matched null agents; stimulus bootstraps; paired comparisons on the generator subset.
Detector validity gate: scenarios where the detector reaches >= 0.75 accuracy on real outcomes form the primary set.
Outputs: results/physion/physion_results.json / .md, figures/*.png
"""

from __future__ import annotations

import argparse
import glob
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from world_maps import physion as P  # noqa: E402
from world_maps.paths import RESULTS  # noqa: E402

warnings.filterwarnings("ignore")
OUT = RESULTS / "physion"
CS = [0.01, 0.1, 1.0, 10.0]


def logit_cv(X, y, groups=None, seed=0, n_splits=5):
    """Cross-validated logistic P(yes). groups=None -> stratified K-fold; else leave-one-group-out."""
    X = np.nan_to_num(np.asarray(X, float))
    y = np.asarray(y).astype(int)
    p = np.full(len(y), np.nan)
    if groups is None:
        folds = list(StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(X, y))
    else:
        folds = [(np.where(groups != g)[0], np.where(groups == g)[0]) for g in np.unique(groups)]
    for tr, te in folds:
        if len(np.unique(y[tr])) < 2:
            p[te] = y[tr].mean()
            continue
        sc = StandardScaler().fit(X[tr])
        best, bestc = -1, 1.0
        for C in CS:
            inner = StratifiedKFold(3, shuffle=True, random_state=seed)
            accs = []
            for itr, ite in inner.split(X[tr], y[tr]):
                m = LogisticRegression(C=C, max_iter=2000).fit(sc.transform(X[tr][itr]), y[tr][itr])
                accs.append((m.predict(sc.transform(X[tr][ite])) == y[tr][ite]).mean())
            if np.mean(accs) > best:
                best, bestc = np.mean(accs), C
        m = LogisticRegression(C=bestc, max_iter=2000).fit(sc.transform(X[tr]), y[tr])
        p[te] = m.predict_proba(sc.transform(X[te]))[:, 1]
    return p


def agreement(p_model, p_human, label, n_boot=1000, seed=0):
    """Accuracy, r with human P(yes), kappa with the human majority, partial r given the label, with stimulus bootstraps."""
    p_model, p_human, label = np.asarray(p_model, float), np.asarray(p_human, float), np.asarray(label).astype(int)
    ok = ~np.isnan(p_model) & ~np.isnan(p_human)
    p_model, p_human, label = p_model[ok], p_human[ok], label[ok]
    if len(label) < 8:
        return {"n": int(len(label))}

    def partial_r(a, b, c):
        ra = a - np.polyval(np.polyfit(c, a, 1), c)
        rb = b - np.polyval(np.polyfit(c, b, 1), c)
        return float(np.corrcoef(ra, rb)[0, 1]) if ra.std() > 0 and rb.std() > 0 else float("nan")

    def m(idx):
        pm, ph, lb = p_model[idx], p_human[idx], label[idx]
        r = float(np.corrcoef(pm, ph)[0, 1]) if pm.std() > 0 and ph.std() > 0 else float("nan")
        return {"accuracy": float(((pm > 0.5).astype(int) == lb).mean()), "r_human": r,
                "kappa_majority": float(cohen_kappa_score((pm > 0.5).astype(int), (ph > 0.5).astype(int))),
                "partial_r_given_label": partial_r(pm, ph, lb.astype(float)),
                "auc": float(roc_auc_score(lb, pm)) if len(np.unique(lb)) > 1 else float("nan")}

    point = m(np.arange(len(label)))
    rng = np.random.default_rng(seed)
    boots = [m(rng.integers(0, len(label), len(label))) for _ in range(n_boot)]
    out = {"n": int(len(label))}
    for k, v in point.items():
        bs = np.array([b[k] for b in boots], float)
        out[k] = v
        out[f"{k}_ci"] = [float(np.nanpercentile(bs, 2.5)), float(np.nanpercentile(bs, 97.5))]
    # accuracy-matched null: random agents with the same accuracy and independent errors
    acc = point["accuracy"]
    null_r, null_k, null_pr = [], [], []
    for _ in range(200):
        correct = rng.random(len(label)) < acc
        pm = np.where(correct, label, 1 - label).astype(float)
        pm = np.clip(pm + rng.normal(0, 0.05, len(pm)), 0, 1)
        null_r.append(np.corrcoef(pm, p_human)[0, 1] if pm.std() > 0 else np.nan)
        null_k.append(cohen_kappa_score((pm > 0.5).astype(int), (p_human > 0.5).astype(int)))
        null_pr.append(partial_r(pm, p_human, label.astype(float)))
    out["null_r_human"], out["null_kappa"], out["null_partial_r"] = float(np.nanmean(null_r)), float(np.nanmean(null_k)), float(np.nanmean(null_pr))
    out["excess_partial_r"] = float(point["partial_r_given_label"] - np.nanmean(null_pr))
    out["_boot_partial"] = np.array([b["partial_r_given_label"] for b in boots], float)
    out["_boot_r"] = np.array([b["r_human"] for b in boots], float)
    return out


def paired_diff(a_boot, b_boot):
    d = a_boot - b_boot
    d = d[~np.isnan(d)]
    if len(d) == 0:
        return None
    return {"diff_mean": float(d.mean()), "diff_ci": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            "p_two_sided": float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))}


def human_reliability(df):
    """Reliability of the per-stimulus human P(yes): 1 - mean binomial sampling variance / observed variance."""
    out = {}
    for scen, g in df.groupby("scenario"):
        p, n = g.p_yes_human.values, g.n_human.values
        sv = np.mean(p * (1 - p) / n)
        out[scen] = float(1 - sv / p.var()) if p.var() > 0 else float("nan")
    p, n = df.p_yes_human.values, df.n_human.values
    out["all"] = float(1 - np.mean(p * (1 - p) / n) / p.var())
    return out


def strip(d):
    return {k: v for k, v in d.items() if not k.startswith("_")}


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=str(OUT))
    ap.add_argument("--n-boot", type=int, default=1000)
    ap.add_argument("--gate", type=float, default=0.75)
    args = ap.parse_args(argv)
    out = Path(args.out)
    (out / "figures").mkdir(parents=True, exist_ok=True)
    stim = pd.read_csv(out / "stimuli.csv")
    stim["label"] = stim.label.astype(str).str.lower().eq("true")
    stim = stim.dropna(subset=["p_yes_human"]).reset_index(drop=True)
    res = {"n_stimuli": int(len(stim)), "human": {}, "detector": {}, "readouts": {}, "generator": {}, "comparisons": {}}

    # ---- humans
    res["human"]["accuracy_by_scenario"] = {k: float(v) for k, v in stim.groupby("scenario").human_acc.mean().items()}
    res["human"]["p_yes_mean"] = float(stim.p_yes_human.mean())
    res["human"]["reliability"] = human_reliability(stim)

    # ---- detector validation and calibration on REAL outcomes
    det = pd.read_csv(out / "detector_real.csv") if (out / "detector_real.csv").exists() else None
    calib = {}
    gate_ok = set(P.SCENARIOS)
    if det is not None:
        det["label"] = det.label.astype(str).str.lower().eq("true")
        det["real_contact"] = det.real_contact.map(lambda v: np.nan if pd.isna(v) else str(v).lower() == "true")
        by = {}
        for scen, g in det.groupby("scenario"):
            ok = g.real_contact.notna()
            acc = float((g.real_contact[ok] == g.label[ok]).mean()) if ok.any() else float("nan")
            feats = g[["real_score", "real_max_overlap", "real_min_dist", "real_base_overlap"]].copy()
            feats["real_min_dist"] = feats.real_min_dist.clip(upper=60)
            feats["vanish"] = g.real_vanish.astype(str).str.lower().eq("true").astype(float)
            pcal = logit_cv(feats.values, g.label.values, seed=0)
            by[scen] = {"rule_accuracy": acc, "rule_auc": float(roc_auc_score(g.label, g.real_score.fillna(0))),
                        "calibrated_cv_accuracy": float(((pcal > 0.5) == g.label.values).mean()), "n": int(len(g)),
                        "frac_valid": float(g.real_frac_valid.mean())}
            sc = StandardScaler().fit(np.nan_to_num(feats.values))
            calib[scen] = (sc, LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(np.nan_to_num(feats.values)), g.label.values.astype(int)))
        res["detector"]["by_scenario"] = by
        gate_ok = {s for s, v in by.items() if max(v["rule_accuracy"], v["calibrated_cv_accuracy"]) >= args.gate}
        res["detector"]["primary_scenarios"] = sorted(gate_ok)
        res["detector"]["overall_rule_accuracy"] = float((det.real_contact == det.label)[det.real_contact.notna()].mean())

    # ---- readouts on the full set
    preds = {}
    kin = pd.read_csv(out / "kinematics.csv") if (out / "kinematics.csv").exists() else None
    if kin is not None:
        k = stim[["stim_id", "scenario", "label"]].merge(kin, on="stim_id", how="left")
        X = k[[c for c in k.columns if c.startswith("k")]].values
        preds["kinematic/within"] = pd.Series(logit_cv(X, k.label.values), index=k.stim_id)
        preds["kinematic/loso"] = pd.Series(logit_cv(X, k.label.values, groups=k.scenario.values), index=k.stim_id)
    for f in sorted(glob.glob(str(out / "features" / "*.npz"))):
        name = os.path.basename(f)[:-4]
        z = np.load(f)
        ids = z["stim_id"]
        order = pd.Series(np.arange(len(ids)), index=ids).reindex(stim.stim_id).values
        okrow = ~pd.isna(order)
        for key in [k for k in z.files if k.startswith("X_")]:
            Xf = z[key].astype(np.float32)[order[okrow].astype(int)]
            Xp = PCA(min(128, Xf.shape[0] - 1, Xf.shape[1]), random_state=0).fit_transform(np.nan_to_num(Xf))
            lab = stim.label.values[okrow]
            s_w = pd.Series(np.nan, index=stim.stim_id)
            s_w.iloc[np.where(okrow)[0]] = logit_cv(Xp, lab)
            s_l = pd.Series(np.nan, index=stim.stim_id)
            s_l.iloc[np.where(okrow)[0]] = logit_cv(Xp, lab, groups=stim.scenario.values[okrow])
            preds[f"{name}/{key[2:]}/within"] = s_w
            preds[f"{name}/{key[2:]}/loso"] = s_l
    if (out / "vlm.csv").exists():
        v = pd.read_csv(out / "vlm.csv").drop_duplicates("stim_id").set_index("stim_id").p_yes
        preds["vlm"] = v.reindex(stim.stim_id)

    def evaluate(series, subset=None):
        d = stim.set_index("stim_id")
        idx = series.index if subset is None else [i for i in series.index if i in subset]
        d = d.loc[idx]
        pm = series.loc[idx].values
        allres = {"all": agreement(pm, d.p_yes_human.values, d.label.values, n_boot=args.n_boot)}
        prim = [i for i in idx if d.loc[i, "scenario"] in gate_ok]
        allres["primary"] = agreement(series.loc[prim].values, d.loc[prim].p_yes_human.values, d.loc[prim].label.values, n_boot=args.n_boot)
        allres["by_scenario"] = {scen: strip(agreement(series.loc[g.index].values, g.p_yes_human.values, g.label.values, n_boot=100))
                                 for scen, g in d.groupby("scenario")}
        return allres

    for name, s in preds.items():
        res["readouts"][name] = {k: (strip(v) if isinstance(v, dict) and "_boot_partial" in v else v) for k, v in evaluate(s).items()}

    # ---- generators
    gen_boot = {}
    for sf in sorted(glob.glob(str(out / "generated" / "*" / "scores.csv"))):
        gname = Path(sf).parent.name
        g = pd.read_csv(sf)
        g["label"] = g.label.astype(str).str.lower().eq("true")
        feats = g[["gen_score", "gen_max_overlap", "gen_min_dist", "gen_base_overlap"]].copy()
        feats["gen_min_dist"] = feats.gen_min_dist.clip(upper=60)
        feats["vanish"] = g.gen_vanish.astype(str).str.lower().eq("true").astype(float)
        pcal = np.full(len(g), np.nan)
        for scen in g.scenario.unique():
            sel = (g.scenario == scen).values
            if scen in calib:
                sc, m = calib[scen]
                pcal[sel] = m.predict_proba(sc.transform(np.nan_to_num(feats.values[sel])))[:, 1]
            else:
                pcal[sel] = g.gen_contact[sel].astype(str).str.lower().eq("true").astype(float)
        g["p_cal"] = pcal
        g["p_rule"] = g.gen_contact.astype(str).str.lower().eq("true").astype(float)
        per = g.groupby("stim_id").agg(p_cal=("p_cal", "mean"), p_rule=("p_rule", "mean"), k=("sample", "size"))
        subset = set(per.index)
        entry = {"n_stimuli": int(len(per)), "samples_per_stimulus": float(per.k.mean()), "n_failed_detection": int(g.gen_contact.isna().sum())}
        for key in ["p_cal", "p_rule"]:
            s = per[key].reindex([i for i in stim.stim_id if i in subset])
            ev = evaluate(s, subset=subset)
            gen_boot[(gname, key)] = ev
            entry[key] = {k: (strip(v) if isinstance(v, dict) and "_boot_partial" in v else v) for k, v in ev.items()}
        # the other readouts on the same subset, for paired comparisons
        comp = {}
        for name, s in preds.items():
            ev = evaluate(s, subset=subset)
            comp[name] = {k: (strip(v) if isinstance(v, dict) and "_boot_partial" in v else v) for k, v in ev.items()}
            for level in ["all", "primary"]:
                a, b = gen_boot[(gname, "p_cal")][level], ev[level]
                if "_boot_partial" in a and "_boot_partial" in b:
                    res["comparisons"][f"{gname} vs {name} [{level}]"] = {
                        "partial_r": paired_diff(a["_boot_partial"], b["_boot_partial"]), "r_human": paired_diff(a["_boot_r"], b["_boot_r"])}
        entry["others_on_subset"] = comp
        res["generator"][gname] = entry

    # ---- scenario profile correlation with humans (accuracy by scenario)
    hum_prof = pd.Series(res["human"]["accuracy_by_scenario"])
    res["scenario_profile_r"] = {}
    for name, r in res["readouts"].items():
        prof = pd.Series({k: v.get("accuracy", np.nan) for k, v in r["by_scenario"].items()}).reindex(hum_prof.index)
        if prof.notna().sum() >= 5:
            res["scenario_profile_r"][name] = float(stats.spearmanr(prof.values, hum_prof.values, nan_policy="omit").correlation)

    figures(res, out / "figures")
    with open(out / "physion_results.json", "w") as f:
        json.dump(res, f, indent=1, default=lambda o: None if isinstance(o, float) and np.isnan(o) else (o.item() if hasattr(o, "item") else str(o)))
    md = summarize(res)
    (out / "physion_results.md").write_text(md)
    print(md)


def summarize(res):
    f = lambda v, d=2: "–" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{v:.{d}f}"
    L = ["# Physion: generators as simulators vs readouts, against human predictions", ""]
    L.append(f"Stimuli with human data: {res['n_stimuli']}. Human P(yes) reliability (binomial-corrected): all {f(res['human']['reliability'].get('all'))}; "
             f"human accuracy by scenario: " + ", ".join(f"{k} {v:.2f}" for k, v in res["human"]["accuracy_by_scenario"].items()))
    if res["detector"]:
        L.append("Detector on real outcomes (rule / calibrated CV accuracy): " + ", ".join(
            f"{k} {v['rule_accuracy']:.2f}/{v['calibrated_cv_accuracy']:.2f}" for k, v in res["detector"]["by_scenario"].items())
                 + f". Primary scenarios (>= gate): {', '.join(res['detector']['primary_scenarios'])}.")
    L += ["", "## Readouts on all stimuli", "", "| readout | n | accuracy | r with human P(yes) [CI] | κ vs majority | partial r given label [CI] (null) | primary: partial r [CI] |", "|---|---|---|---|---|---|---|"]
    for name, r in res["readouts"].items():
        a, p = r["all"], r["primary"]
        L.append(f"| {name} | {a.get('n')} | {f(a.get('accuracy'))} | {f(a.get('r_human'))} [{f(a.get('r_human_ci',[None,None])[0])}, {f(a.get('r_human_ci',[None,None])[1])}] | "
                 f"{f(a.get('kappa_majority'))} | {f(a.get('partial_r_given_label'))} [{f(a.get('partial_r_given_label_ci',[None,None])[0])}, {f(a.get('partial_r_given_label_ci',[None,None])[1])}] ({f(a.get('null_partial_r'))}) | "
                 f"{f(p.get('partial_r_given_label'))} [{f(p.get('partial_r_given_label_ci',[None,None])[0])}, {f(p.get('partial_r_given_label_ci',[None,None])[1])}] |")
    for gname, g in res["generator"].items():
        L += ["", f"## Generator: {gname} ({g['n_stimuli']} stimuli, {g['samples_per_stimulus']:.1f} samples each)", "",
              "| readout (same subset) | accuracy | r with human P(yes) [CI] | κ | partial r given label [CI] (null) | primary: partial r [CI] |", "|---|---|---|---|---|---|"]
        for key in ["p_cal", "p_rule"]:
            a, p = g[key]["all"], g[key]["primary"]
            L.append(f"| **{gname} {key}** | {f(a.get('accuracy'))} | {f(a.get('r_human'))} [{f(a.get('r_human_ci',[None,None])[0])}, {f(a.get('r_human_ci',[None,None])[1])}] | {f(a.get('kappa_majority'))} | "
                     f"{f(a.get('partial_r_given_label'))} [{f(a.get('partial_r_given_label_ci',[None,None])[0])}, {f(a.get('partial_r_given_label_ci',[None,None])[1])}] ({f(a.get('null_partial_r'))}) | "
                     f"{f(p.get('partial_r_given_label'))} [{f(p.get('partial_r_given_label_ci',[None,None])[0])}, {f(p.get('partial_r_given_label_ci',[None,None])[1])}] |")
        for name, r in g["others_on_subset"].items():
            a, p = r["all"], r["primary"]
            L.append(f"| {name} | {f(a.get('accuracy'))} | {f(a.get('r_human'))} [{f(a.get('r_human_ci',[None,None])[0])}, {f(a.get('r_human_ci',[None,None])[1])}] | {f(a.get('kappa_majority'))} | "
                     f"{f(a.get('partial_r_given_label'))} [{f(a.get('partial_r_given_label_ci',[None,None])[0])}, {f(a.get('partial_r_given_label_ci',[None,None])[1])}] ({f(a.get('null_partial_r'))}) | "
                     f"{f(p.get('partial_r_given_label'))} [{f(p.get('partial_r_given_label_ci',[None,None])[0])}, {f(p.get('partial_r_given_label_ci',[None,None])[1])}] |")
        L += ["", "Paired comparisons (generator calibrated P(yes) minus readout; stimulus bootstrap):"]
        for k, v in res["comparisons"].items():
            if k.startswith(gname) and v.get("partial_r"):
                L.append(f"- {k}: partial r diff {v['partial_r']['diff_mean']:+.3f} [{v['partial_r']['diff_ci'][0]:+.3f}, {v['partial_r']['diff_ci'][1]:+.3f}] (p = {v['partial_r']['p_two_sided']:.3f}); "
                         f"r diff {v['r_human']['diff_mean']:+.3f} [{v['r_human']['diff_ci'][0]:+.3f}, {v['r_human']['diff_ci'][1]:+.3f}]")
    if res.get("scenario_profile_r"):
        L += ["", "Scenario-profile correlation with human accuracy (Spearman over 8 scenarios): " + ", ".join(f"{k} {v:.2f}" for k, v in res["scenario_profile_r"].items())]
    L += ["", "partial r given label: correlation between model and human P(yes) after the ground-truth label is regressed out of both, i.e. agreement on *which* stimuli are judged wrongly. "
          "null = accuracy-matched random agent. Primary = scenarios where the contact detector passed the validity gate on real outcomes."]
    return "\n".join(L)


def figures(res, out: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    rows = []
    for name, r in res["readouts"].items():
        rows.append((name, r["all"].get("partial_r_given_label"), r["all"].get("partial_r_given_label_ci", [np.nan, np.nan]), r["all"].get("accuracy"), "#36c"))
    for gname, g in res["generator"].items():
        a = g["p_cal"]["all"]
        rows.append((f"{gname} (imagined outcome)", a.get("partial_r_given_label"), a.get("partial_r_given_label_ci", [np.nan, np.nan]), a.get("accuracy"), "#2a2"))
    rows = [r for r in rows if r[1] is not None and np.isfinite(r[1])]
    if not rows:
        return
    rows.sort(key=lambda r: -r[1])
    fig, ax = plt.subplots(figsize=(max(6, 0.45 * len(rows) + 3), 4.5))
    x = np.arange(len(rows))
    ax.bar(x, [r[1] for r in rows], yerr=[[r[1] - r[2][0] for r in rows], [r[2][1] - r[1] for r in rows]], color=[r[4] for r in rows])
    ax.axhline(0, c="gray", lw=0.5)
    ax.set_xticks(x); ax.set_xticklabels([f"{r[0]}\nacc {r[3]:.2f}" for r in rows], rotation=70, ha="right", fontsize=6)
    ax.set_ylabel("partial r with human P(yes) | label"); ax.set_title("Which readout gets the same stimuli wrong as people?")
    fig.tight_layout(); fig.savefig(out / "error_agreement.png", dpi=150); plt.close(fig)
    # scenario heatmap of accuracy
    names = list(res["readouts"]) + [f"{g} (imagined)" for g in res["generator"]]
    scen = list(res["human"]["accuracy_by_scenario"])
    M = np.full((len(names) + 1, len(scen)), np.nan)
    M[0] = [res["human"]["accuracy_by_scenario"][s] for s in scen]
    for i, n in enumerate(list(res["readouts"])):
        M[i + 1] = [res["readouts"][n]["by_scenario"].get(s, {}).get("accuracy", np.nan) for s in scen]
    for j, g in enumerate(res["generator"]):
        M[len(res["readouts"]) + 1 + j] = [res["generator"][g]["p_cal"]["by_scenario"].get(s, {}).get("accuracy", np.nan) for s in scen]
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(names) + 2))
    im = ax.imshow(M, vmin=0.4, vmax=1.0, cmap="viridis", aspect="auto")
    ax.set_yticks(range(len(names) + 1)); ax.set_yticklabels(["humans"] + names, fontsize=6)
    ax.set_xticks(range(len(scen))); ax.set_xticklabels(scen, rotation=45, ha="right", fontsize=7)
    plt.colorbar(im, ax=ax, label="accuracy"); fig.tight_layout(); fig.savefig(out / "accuracy_by_scenario.png", dpi=150); plt.close(fig)


if __name__ == "__main__":
    main()


In [ ]:
#@title 3. Data (270 MB) and detector validation on the real outcomes (CPU; skipped when the committed results exist)
%cd /content/world-maps
!PYTHONPATH=src python scripts/physion_extract.py --stages detector 2>&1 | tail -14


In [ ]:
#@title 4. Smoke test: one generated continuation, shown
%cd /content/world-maps
if SMOKE_TEST_FIRST and RUN_GENERATION:
    !PYTHONPATH=src python scripts/physion_generate.py --generator {GENERATOR} --per-scenario 2 --k 1 --size {GEN_SIZE} --n-frames {N_FRAMES} --limit 1 --out /content/smoke_gen 2>&1 | tail -5
    import glob, pandas as pd
    from IPython.display import display, Video
    for f in sorted(glob.glob("/content/smoke_gen/*/*.mp4"))[:1]:
        print(f); display(Video(f, embed=True, width=320))
    for f in glob.glob("/content/smoke_gen/*/scores.csv"):
        display(pd.read_csv(f).T)
else:
    print("skipped")


In [ ]:
#@title 5. Encoders on the 1.5 s context (about 15 min) and VLM (optional, about 20 min)
%cd /content/world-maps
args = " ".join(MODELS)
!CUDA_LAUNCH_BLOCKING=1 PYTHONPATH=src python scripts/physion_extract.py --stages encoders --models {args} 2>&1 | grep -v -i "warn\|Loading weights\|Fetching\|UNEXPECTED" | tail -8
if RUN_VLM:
    !PYTHONPATH=src python scripts/physion_extract.py --stages vlm --vlm-id {VLM_ID} 2>&1 | grep -v -i "warn\|Loading weights\|Fetching" | tail -3


In [ ]:
#@title 6. Generation for the stratified subset (long; resumable)
%cd /content/world-maps
if RUN_GENERATION:
    !PYTHONPATH=src python scripts/physion_generate.py --generator {GENERATOR} --per-scenario {PER_SCENARIO} --k {K_SAMPLES} --size {GEN_SIZE} --n-frames {N_FRAMES} 2>&1 | tail -12
    import glob, numpy as np
    from PIL import Image
    from IPython.display import display
    frames = sorted(glob.glob(f"results/physion/generated/{GENERATOR}/frames/*_last.png"))[:12]
    if frames:
        sheet = np.concatenate([np.array(Image.open(f)) for f in frames], 1)
        Image.fromarray(sheet).save(f"results/physion/generated/{GENERATOR}/last_frames_sheet.png"); display(Image.fromarray(sheet))
else:
    print("skipped")


In [ ]:
#@title 7. Analysis (CPU, about 5 min)
%cd /content/world-maps
!PYTHONPATH=src python scripts/physion_analyze.py --n-boot 1000
from IPython.display import Image as IM, display
for f in ["error_agreement.png", "accuracy_by_scenario.png"]:
    display(IM(filename=f"results/physion/figures/{f}"))


In [ ]:
#@title 8. Package results for download
%cd /content/world-maps
!zip -q -r /content/world_maps_physion_results.zip results/physion/physion_results.json results/physion/physion_results.md results/physion/figures results/physion/detector_real.csv results/physion/kinematics.csv results/physion/stimuli.csv results/physion/vlm.csv results/physion/features/models.json $(ls results/physion/generated/*/scores.csv results/physion/generated/*/last_frames_sheet.png 2>/dev/null) $(ls results/physion/generated/*/*.mp4 2>/dev/null | head -8) 2>/dev/null
!ls -la /content/world_maps_physion_results.zip
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/world_maps_physion && cp -r results/physion/features results/physion/generated /content/drive/MyDrive/world_maps_physion/ 2>/dev/null; ls /content/drive/MyDrive/world_maps_physion
from google.colab import files
files.download("/content/world_maps_physion_results.zip")


### If something fails
- **LTX conditioning errors:** `GEN_SIZE` must be a multiple of 32 and `N_FRAMES` = 8k + 1 (97, 121). The context is resampled to 24 fps (36 frames → 33 used).
- **Cosmos:** experimental; needs the `cosmos_guardrail` package and Hub access. Use `ltx` if it fails.
- **Generation too slow:** lower `PER_SCENARIO` to 24 (192 clips) or `N_FRAMES` to 73; the analysis uses whatever exists.
- **Runtime disconnects:** rerun cells 2 and 6; finished clips are skipped (scores.csv is appended).
